# Example: Fun with Iterative Linear Algebraic Equation Solvers
How do iterative solutions compare with a direct solution of the same linear system? We will construct a test problem with known convergence properties, verify a selected iterative method, and compare runtime and memory allocations.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> - **Construct a test system:** Generate a reproducible, strictly diagonally dominant matrix and verify the assumptions needed for convergence.
> - **Validate an iterative solution:** Distinguish the equation residual from the difference relative to a direct solution, and check both quantities.
> - **Compare computational cost:** Benchmark the validated solver configuration and interpret runtime and allocations in light of the implementation.

In this example, we use the Jacobi, Gauss–Seidel, and successive over-relaxation methods introduced in [the lecture](CHEME-5800-L6c-Lecture-GeneralIterativeMethod-Fall-2026.ipynb). Change the selected method to repeat the comparison on the same system.

___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#Base.include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course library and the packages used here.

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"));

See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5800 course library documentation](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/) for the functions and types used here. The [BenchmarkTools manual](https://juliaci.github.io/BenchmarkTools.jl/stable/manual/) describes the timing and allocation measurements used in Task 3.

___


## Task 1: Build a reproducible test system
In this task, we will construct a random square system whose matrix satisfies the convergence assumptions for all three methods.

> __Strict row diagonal dominance:__
>
> For a system with $n$ unknowns, its matrix $\mathbf{A}\in\mathbb{R}^{n\times n}$ is strictly diagonally dominant by rows when:
>
> $$
> |a_{ii}|>\sum_{j\ne i}|a_{ij}|,\qquad i=1,\ldots,n.
> $$
>
> Every row must satisfy this inequality. It guarantees convergence of Jacobi and Gauss–Seidel from any initial guess.

For the relaxation method, we will also make the matrix symmetric with positive diagonal entries. Together with strict diagonal dominance, these properties make it positive definite, so successive over-relaxation converges for any relaxation factor $0<\omega<2$.

Choose the number of unknowns, random seed, and positive diagonal margin:

In [2]:
number_of_rows = 100; # number of equations and unknowns
random_seed = 5800; # reproduce the same system in this Julia environment
diagonal_margin = 1.0; # amount by which each diagonal exceeds its off-diagonal row sum

@assert number_of_rows > 0 "Choose at least one unknown."
@assert isfinite(diagonal_margin) && diagonal_margin > 0 "Use a positive, finite diagonal margin."

We draw a random matrix with [the `randn(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Base.randn), average it with its transpose, and set its diagonal to zero. Call the resulting symmetric matrix $\mathbf{S}$. With positive margin $\delta$, define the system entries by:

$$
\begin{aligned}
a_{ij}&=s_{ij}, &&i\ne j,\\
a_{ii}&=\delta+\sum_{j\ne i}|s_{ij}|, &&i=1,\ldots,n.
\end{aligned}
$$

This construction makes every diagonal entry positive and strictly larger than its off-diagonal row sum. We use a local [Mersenne Twister generator](https://docs.julialang.org/en/v1/stdlib/Random/#Random.MersenneTwister) so rerunning the cell reproduces the matrix and right-hand side without resetting the notebook's global random stream.

In [3]:
A, b = let
    # Generate symmetric off-diagonal entries -
    rng = MersenneTwister(random_seed); # local random-number generator
    random_matrix = randn(rng, number_of_rows, number_of_rows);
    matrix = (random_matrix + transpose(random_matrix)) / 2;
    for i in 1:number_of_rows
        matrix[i, i] = 0.0;
    end

    # Enforce strict diagonal dominance in every row -
    for i in 1:number_of_rows
        matrix[i, i] = sum(abs, matrix[i, :]) + diagonal_margin;
    end
    rhs = randn(rng, number_of_rows); # dimensionless test right-hand side

    matrix, rhs
end;

Inspect the leading block of the matrix. The full row, including entries outside this preview, determines diagonal dominance.

In [4]:
A[1:min(5, number_of_rows), 1:min(5, number_of_rows)]

5×5 Matrix{Float64}:
 64.8653     0.220313   0.152694      0.409255  -0.298376
  0.220313  56.9269     0.208682      0.452789  -0.425506
  0.152694   0.208682  49.2828        0.43118    0.000108388
  0.409255   0.452789   0.43118      57.6521     0.734767
 -0.298376  -0.425506   0.000108388   0.734767  54.1406

### Check the matrix assumptions
We calculate each row's diagonal margin independently of the construction. A positive margin in every row confirms strict diagonal dominance; the symmetry and positive-definiteness checks confirm the additional assumptions used for relaxation.

In [5]:
row_margins = let
    margins = zeros(number_of_rows); # diagonal magnitude minus off-diagonal absolute row sum
    for i in 1:number_of_rows
        off_diagonal_sum = sum(abs(A[i, j]) for j in 1:number_of_rows if j != i; init = 0.0);
        margins[i] = abs(A[i, i]) - off_diagonal_sum;
    end
    margins
end;

ddcondition = row_margins .> 0;
@assert all(ddcondition) "Every row must be strictly diagonally dominant."
@assert issymmetric(A) "The relaxation guarantee used here requires symmetry."
@assert isposdef(A) "The relaxation guarantee used here requires positive definiteness."

In [6]:
(rows_passing = count(ddcondition), total_rows = number_of_rows,
 minimum_margin = minimum(row_margins), symmetric = issymmetric(A), positive_definite = isposdef(A))

(rows_passing = 100, total_rows = 100, minimum_margin = 0.9999999999999503, symmetric = true, positive_definite = true)

The number of passing rows should equal the matrix dimension, and the minimum margin should agree with the chosen value up to floating-point rounding. The assertions stop execution if the assumptions fail; checking only one successful row would not establish diagonal dominance.

___

## Task 2: Solve and validate the system
In this task, we will solve $\mathbf{A}\mathbf{x}=\mathbf{b}$ with a selected iterative method and compare it with Julia's direct solution. The [backslash operator](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#Base.:\-Tuple{AbstractMatrix,%20AbstractVecOrMat}) supplies our reference solution:

In [7]:
julia_solution = A \ b; # direct reference solution

Set the iterative method and stopping parameters once. Task 3 will reuse these values so the benchmark measures the configuration we validate here. The relaxation factor is used only by the successive over-relaxation method; for that method, values below one give under-relaxation and values above one give over-relaxation.

In [8]:
algorithm = GaussSeidelMethod(); # alternatives: JacobiMethod() or SuccessiveOverRelaxationMethod()
maximum_number_of_iterations = 1000; # requested correction limit
residual_tolerance = 1e-10; # absolute Euclidean residual tolerance
solution_check_tolerance = 1e-8; # absolute difference allowed relative to the direct solution
ω = 1.1; # relaxation factor, used only by SuccessiveOverRelaxationMethod()
xₒ = 0.1 * ones(number_of_rows); # same initial guess for validation and benchmarking

@assert maximum_number_of_iterations > 0;
@assert residual_tolerance > 0 && solution_check_tolerance > 0;
@assert 0 < ω < 2 "Use a relaxation factor between zero and two."

[The `solve(...)` function](../../../code/src/Solvers.jl) returns a dictionary containing the initial guess at key zero and the stored iterates at later integer keys. Returning an archive does not establish convergence, so we will check the final stored vector explicitly.

In [9]:
our_solution_archive = VLDataScienceMachineLearningPackage.solve(A, b, xₒ;
    algorithm = algorithm,
    maxiterations = maximum_number_of_iterations,
    ϵ = residual_tolerance,
    ω = ω,
);

### Check the residual and solution difference
The residual measures how well the iterative solution satisfies the equations. The difference from the direct solution measures agreement between the two numerical answers. For the final stored vector $\widehat{\mathbf{x}}$ and direct solution $\mathbf{x}_{\mathrm{direct}}$, these quantities are:

$$
\begin{aligned}
\mathbf{r}&=\mathbf{b}-\mathbf{A}\widehat{\mathbf{x}}, &&\text{equation residual},\\
\Delta\mathbf{x}&=\widehat{\mathbf{x}}-\mathbf{x}_{\mathrm{direct}}, &&\text{solution difference}.
\end{aligned}
$$

We measure both using [the `norm(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.norm). Its default vector norm is the Euclidean norm, defined for a vector $\mathbf{v}\in\mathbb{R}^n$ by:

$$
\|\mathbf{v}\|_2=\sqrt{\sum_{i=1}^n v_i^2}.
$$

The residual must meet the solver's tolerance before we interpret its runtime. Agreement with the direct solve supplies a separate check for this test system; a small residual alone is not a universal bound on solution error.

In [10]:
last_iteration = maximum(keys(our_solution_archive)); # archive keys count stored corrections
last_solution = our_solution_archive[last_iteration];
residual_norm = norm(b - A * last_solution);
solution_difference_norm = norm(last_solution - julia_solution);
direct_residual_norm = norm(b - A * julia_solution);

@assert all(isfinite, last_solution) "The iterative solution contains a nonfinite entry."
@assert residual_norm < residual_tolerance "The final stored iterate did not meet the residual tolerance."
@assert last_iteration <= maximum_number_of_iterations "The stored corrections exceeded the requested limit."
@assert solution_difference_norm <= solution_check_tolerance "The solutions disagree beyond the comparison tolerance."

DataFrame(
    method = ["Direct", string(nameof(typeof(algorithm)))],
    residual_norm = [direct_residual_norm, residual_norm],
    difference_from_direct = [0.0, solution_difference_norm],
)

Row,method,residual_norm,difference_from_direct
,String,Float64,Float64
1,Direct,5.49485e-15,0.0
2,GaussSeidelMethod,7.22805e-12,1.40829e-13


The residual column tests the equations, while the difference column compares solutions. Passing these checks validates this run; it does not establish correctness for every matrix. After changing the method, tolerance, or initial guess, rerun Task 2 before benchmarking.

The solver checks the residual before taking another correction. It returns immediately when the tolerance is met or the correction limit is reached. The largest archive key counts completed corrections; the archive has one additional entry for the initial guess. Reaching the limit alone does not establish convergence, so we check the final residual explicitly.

___

## Task 3: Compare runtime and memory allocations
In this task, we will benchmark the direct solve and the validated iterative configuration on the same matrix and right-hand side. [The `@benchmark` macro](https://juliaci.github.io/BenchmarkTools.jl/stable/manual/#Benchmarking-basics) repeats each calculation and reports a distribution of timings and allocation estimates.

Both measurements include solver setup. The direct call computes a factorization and returns one solution; the course iterative call constructs explicit inverse matrices and retains its solution history. These implementation choices affect the measured cost. This comparison does not represent the cost of every implementation of these algorithms.

We interpolate the inputs into each benchmark and use one evaluation per sample. Begin with the direct solve:

In [11]:
direct_trial = @benchmark $A \ $b seconds = 2 samples = 500 evals = 1

BenchmarkTools.Trial: 500 samples with 1 evaluation per sample.
 Range (min … max):  37.042 μs … 92.333 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     41.833 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   42.708 μs ±  3.914 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

      ▁          ▁█▅           ▁▃                              
  ▅▂▁▄█▄▂▂▁▂▁▃▆▅▄███▅▃▂▂▂▃▂▃▃▁▄██▅▃▂▃▂▂▃▃▃▁▂▂▃▁▃▁▂▁▁▁▃▂▂▁▁▁▁▂ ▃
  37 μs           Histogram: frequency by time        53.5 μs <

 Memory estimate: 81.89 KiB, allocs estimate: 7.

Now benchmark the same iterative call used in Task 2, including the same method, initial guess, residual tolerance, correction limit, and relaxation factor:

In [12]:
iterative_trial = @benchmark VLDataScienceMachineLearningPackage.solve($A, $b, $xₒ;
    algorithm = $algorithm,
    maxiterations = $maximum_number_of_iterations,
    ϵ = $residual_tolerance,
    ω = $ω,
) seconds = 2 samples = 500 evals = 1

BenchmarkTools.Trial: 500 samples with 1 evaluation per sample.
 Range (min … max):  57.083 μs … 577.917 μs  ┊ GC (min … max): 0.00% … 81.49%
 Time  (median):     90.062 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   91.963 μs ±  36.011 μs  ┊ GC (mean ± σ):  1.02% ±  3.64%

      ▆█▄▂▆▃   ▄█▆▄▃▂                                           
  █▄▁▁██████▇█▇███████▆▆▆▆▄▁▄▁▄▁▅▇▄▄▁▅▄▁▁▄▆▄▁▁▁▅▄▁▄▁▄▁▁█▆▄▆▄▄▅ █
  57.1 μs       Histogram: log(frequency) by time       199 μs <

 Memory estimate: 445.52 KiB, allocs estimate: 119.

Compare the median time per solve and the allocated bytes and allocation counts reported by [the benchmark estimates](https://juliaci.github.io/BenchmarkTools.jl/stable/manual/#Handling-benchmark-results). Allocated memory measures allocation traffic during a call, not peak memory in use.

In [13]:
benchmark_comparison = let
    direct_estimate = BenchmarkTools.median(direct_trial);
    iterative_estimate = BenchmarkTools.median(iterative_trial);
    estimates = [direct_estimate, iterative_estimate];

    DataFrame(
        method = ["Direct", string(nameof(typeof(algorithm)))],
        median_time_ms = [estimate.time / 1e6 for estimate in estimates],
        allocated_KiB = [estimate.memory / 1024 for estimate in estimates],
        allocations = [estimate.allocs for estimate in estimates],
    )
end

Row,method,median_time_ms,allocated_KiB,allocations
,String,Float64,Float64,Int64
1,Direct,0.041833,81.8906,7
2,GaussSeidelMethod,0.090062,445.516,119


### What should we compare?
Use the measured median times to identify the faster call for the current problem. Compare allocations separately: the iterative archive contains a vector for every stored correction, while the direct solver returns one vector. Constructing the matrix splitting and explicit inverses also contributes to the iterative call's allocations and runtime.

The ranking can change with the matrix dimension, conditioning, method, tolerance, and relaxation factor. Timing also depends on the machine and its current workload. Change one parameter at a time, rerun the correctness checks, and repeat both benchmarks. Do not infer a universal speed advantage from one system.

___

## Summary
We constructed a test system, checked an iterative solution, and compared the cost of the validated call with a direct solve.

> __Key Takeaways:__
>
> - **Matrix construction and convergence:** We constructed a symmetric, strictly diagonally dominant matrix with positive diagonal entries and checked every row. This gave us a test problem with convergence guarantees for all three methods over the stated relaxation interval.
> - **Residual and solution checks:** We distinguished the equation residual from the difference relative to the direct solution. Checking both prevented a completed solver call from being mistaken for a successful solve.
> - **Runtime and memory comparisons:** We reused the same iterative configuration for correctness checks and timing. We interpreted runtime and allocations in terms of the matrix, stopping tolerance, explicit inverses, and retained solution history.

These checks provide a basis for comparing methods as we change the test system and solver parameters.

___